In [1]:
import torch
print("GPU available:", torch.cuda.is_available())

!pip install jiwer
!pip install peft
!pip install evaluate

GPU available: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 65.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.3 MB/s eta 0:00:00:00:0100:0

In [2]:
# Try importing with fallbacks
import warnings
warnings.filterwarnings("ignore")

try:
  from transformers import (
      WhisperFeatureExtractor,
      WhisperTokenizer,
      WhisperProcessor,
      WhisperForConditionalGeneration,
      Seq2SeqTrainingArguments,
      Seq2SeqTrainer
  )
  print("✅ Transformers imported successfully")
except Exception as e:
  print(f"❌ Import error: {e}")
  print("Trying alternative approach...")

  # Alternative: use older API
  from transformers import pipeline
  print("Using pipeline fallback")

# Test if basic functionality works
try:
  feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
  print("✅ Whisper components working")
except Exception as e:
  print(f"❌ Whisper error: {e}")

2025-08-24 23:21:12.027918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756077672.225975      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756077672.280123      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Transformers imported successfully


preprocessor_config.json: 0.00B [00:00, ?B/s]

✅ Whisper components working


In [3]:
import os
from datasets import load_from_disk

# Verify dataset exists and structure
dataset_path = "/kaggle/input/combined-speech/combined_nigerian_speech"
print(f"🔍 Checking dataset at: {dataset_path}")

if os.path.exists(dataset_path):
  print("✅ Dataset found!")

  # Load and inspect dataset
  dataset = load_from_disk(dataset_path)
  print(f"📊 Dataset structure: {dataset}")

  for split, data in dataset.items():
      print(f"  {split}: {len(data)} samples")
      if "dataset_source" in data.column_names:
          sources = data["dataset_source"]
          print(f"    - Pidgin: {sources.count('pidgin')}")
          print(f"    - Common Voice: {sources.count('common_voice')}")
          print(f"    - Accented English: {sources.count('accented_english')}")
else:
  print("❌ Dataset not found!")

🔍 Checking dataset at: /kaggle/input/combined-speech/combined_nigerian_speech
✅ Dataset found!
📊 Dataset structure: DatasetDict({
    train: Dataset({
        features: ['text', 'filename', 'audio', 'dataset_source', 'client_id', 'path', 'accent', 'locale', 'segment'],
        num_rows: 7060
    })
    validation: Dataset({
        features: ['text', 'filename', 'audio', 'dataset_source', 'client_id', 'path', 'accent', 'locale', 'segment'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['text', 'filename', 'audio', 'dataset_source', 'client_id', 'path', 'accent', 'locale', 'segment'],
        num_rows: 1438
    })
})
  train: 7060 samples
    - Pidgin: 2708
    - Common Voice: 2176
    - Accented English: 2176
  validation: 1221 samples
    - Pidgin: 677
    - Common Voice: 272
    - Accented English: 272
  test: 1438 samples
    - Pidgin: 892
    - Common Voice: 273
    - Accented English: 273


In [3]:
print("🔍 Ultra-minimal test...")
import torch
import numpy as np
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

try:
  # Test 1: Load model fresh
  print("1️⃣ Loading fresh model...")
  model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

  lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "v_proj"])
  model = get_peft_model(model, lora_config)
  print("✅ Model + LoRA OK")

  # Test 2: Processors
  print("2️⃣ Loading processors...")
  feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
  tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", task="transcribe")
  print("✅ Processors OK")

  # Test 3: Data processing
  print("3️⃣ Testing data processing...")
  audio = np.random.randn(16000) * 0.01
  inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
  print("✅ Data processing OK")

  # Test 4: GPU move (critical test)
  print("4️⃣ Moving to GPU...")
  print(f"GPU memory before: {torch.cuda.memory_allocated()/1e9:.2f}GB")
  model = model.cuda()
  print(f"GPU memory after: {torch.cuda.memory_allocated()/1e9:.2f}GB")
  print("✅ GPU move OK")

  # Test 5: Forward pass
  print("5️⃣ Forward pass...")
  model.eval()
  with torch.no_grad():
      inputs_gpu = inputs.to('cuda')
      outputs = model(**inputs_gpu)
      print(f"✅ Forward pass OK, loss: {outputs.loss}")

except Exception as e:
  print(f"❌ Failed: {e}")
# Test 6: Fixed forward pass with proper Whisper inputs
print("6️⃣ Fixed forward pass...")
try:
  model.eval()
  with torch.no_grad():
      inputs_gpu = inputs.to('cuda')

      # Add decoder_input_ids for Whisper
      decoder_input_ids = torch.tensor([[50258]], device='cuda')  # Start token

      outputs = model(
          input_features=inputs_gpu['input_features'],
          decoder_input_ids=decoder_input_ids
      )
      print(f"✅ Fixed forward pass OK, loss: {outputs.loss}")

  # Test 7: Training mode forward pass
  print("7️⃣ Training mode forward pass...")
  model.train()

  # Prepare proper training inputs
  text = "hello world"
  labels = torch.tensor([tokenizer(text).input_ids], device='cuda')

  outputs = model(
      input_features=inputs_gpu['input_features'],
      labels=labels
  )
  print(f"✅ Training forward pass OK, loss: {outputs.loss}")

  # Test 8: Backward pass
  print("8️⃣ Backward pass...")
  loss = outputs.loss
  loss.backward()
  print("✅ Backward pass OK!")

  print("🎉 Everything works! Ready for full training!")

except Exception as e:
  print(f"❌ Failed at step: {e}")
  import traceback
  traceback.print_exc()


🔍 Ultra-minimal test...
1️⃣ Loading fresh model...
✅ Model + LoRA OK
2️⃣ Loading processors...
✅ Processors OK
3️⃣ Testing data processing...
✅ Data processing OK
4️⃣ Moving to GPU...
GPU memory before: 0.15GB
GPU memory after: 1.13GB
✅ GPU move OK
5️⃣ Forward pass...
❌ Failed: You have to specify either decoder_input_ids or decoder_inputs_embeds
6️⃣ Fixed forward pass...
✅ Fixed forward pass OK, loss: None
7️⃣ Training mode forward pass...
✅ Training forward pass OK, loss: 7.007862091064453
8️⃣ Backward pass...
✅ Backward pass OK!
🎉 Everything works! Ready for full training!


In [4]:
print("🏋️ High-quality training with real audio...")
import torch.optim as optim
import librosa
import numpy as np  # Also add this
from datasets import load_from_disk  # ✅ Add this import
import os

# Better optimizer with scheduling
from torch.optim.lr_scheduler import LinearLR
optimizer = optim.AdamW(model.parameters(), lr=5e-4)
scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=100)  # Warmup

# Load dataset with real audio processing
dataset_path = "/kaggle/input/combined-speech/combined_nigerian_speech"
dataset = load_from_disk(dataset_path)
train_data = dataset["train"].select(range(2000))  # More samples
print(f"✅ Dataset: {len(train_data)} samples")

model.train()
total_loss = 0

for step in range(800):  # More training steps
  sample = train_data[step % len(train_data)]

  # Load REAL audio from dataset
  try:
      if 'audio' in sample and sample['audio'] is not None:
          audio_array = sample['audio']['array']
          if len(audio_array) > 0:
              # Resample to 16kHz if needed
              audio = librosa.resample(audio_array, orig_sr=sample['audio']['sampling_rate'], target_sr=16000)
          else:
              audio = np.random.randn(16000) * 0.01  # Fallback
      else:
          audio = np.random.randn(16000) * 0.01  # Fallback
  except:
      audio = np.random.randn(16000) * 0.01  # Fallback

  # Process audio and text
  inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
  labels = torch.tensor([tokenizer(sample["text"]).input_ids], device='cuda')

  # Training step
  inputs_gpu = inputs.to('cuda')
  outputs = model(input_features=inputs_gpu['input_features'], labels=labels)
  loss = outputs.loss

  # Backward with gradient clipping
  optimizer.zero_grad()
  loss.backward()
  torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Stability
  optimizer.step()
  scheduler.step()

  total_loss += loss.item()

  if step % 50 == 0:
      avg_loss = total_loss / (step + 1)
      print(f"Step {step}: Loss = {loss.item():.4f}, Avg = {avg_loss:.4f}, LR = {scheduler.get_last_lr()[0]:.6f}")

  torch.cuda.empty_cache()

print("🎉 High-quality training completed!")
model.save_pretrained("/kaggle/working/nigerian-whisper-lora-2k")


🏋️ High-quality training with real audio...
✅ Dataset: 2000 samples
Step 0: Loss = 3.2987, Avg = 3.2987, LR = 0.000055
Step 50: Loss = 2.0355, Avg = 3.2949, LR = 0.000279
Step 100: Loss = 1.7943, Avg = 2.5201, LR = 0.000500
Step 150: Loss = 0.6735, Avg = 2.2112, LR = 0.000500
Step 200: Loss = 1.4778, Avg = 1.9955, LR = 0.000500
Step 250: Loss = 0.5271, Avg = 1.8826, LR = 0.000500
Step 300: Loss = 1.7043, Avg = 1.8060, LR = 0.000500
Step 350: Loss = 0.5694, Avg = 1.7180, LR = 0.000500
Step 400: Loss = 1.9326, Avg = 1.6753, LR = 0.000500
Step 450: Loss = 0.9955, Avg = 1.6183, LR = 0.000500
Step 500: Loss = 0.4562, Avg = 1.5755, LR = 0.000500
Step 550: Loss = 0.4865, Avg = 1.5402, LR = 0.000500
Step 600: Loss = 1.4782, Avg = 1.4869, LR = 0.000500
Step 650: Loss = 1.6947, Avg = 1.4552, LR = 0.000500
Step 700: Loss = 0.3991, Avg = 1.4295, LR = 0.000500
Step 750: Loss = 0.9646, Avg = 1.4204, LR = 0.000500
🎉 High-quality training completed!
